In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        "Could not find .env. Open the notebook from the LangChain repository."
    )

load_dotenv(env_path)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(
        "GROQ_API_KEY is missing or empty in the repository .env file."
    )

## Completion model

In [5]:
from langchain_groq import ChatGroq

llmModel = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
)

# chat completion model

In [6]:
chatModel = llmModel

## Prompts and Prompt Templates

In [7]:
# This is for completions model
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt = prompt_template.format(
    adjective="curious",
    topic="Tesla"
)

res = llmModel.invoke(llmModelPrompt)
print(res)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [6]:
# chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)
print(response.content)


Joseph P. Kennedy, the patriarch of the Kennedy family, had a total of nine grandchildren.


# Old way

In [7]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input="Name the children and grandchildren of Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [8]:
response

AIMessage(content='Joseph P. Kennedy and his wife Rose Kennedy had nine children:\n\n1. Joseph P. Kennedy Jr.\n2. John F. Kennedy\n3. Rosemary Kennedy\n4. Kathleen Kennedy\n5. Eunice Kennedy\n6. Patricia Kennedy\n7. Robert F. Kennedy\n8. Jean Kennedy\n9. Edward M. Kennedy\n\nThe grandchildren of Joseph P. Kennedy include the children of his children. Some of the more well-known grandchildren include Caroline Kennedy (daughter of John F. Kennedy), Maria Shriver (daughter of Eunice Kennedy), and Joseph P. Kennedy II (son of Robert F. Kennedy).', response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 33, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-7d8672f3-8e32-405c-ac37-e

In [9]:
print(response.content)

Joseph P. Kennedy and his wife Rose Kennedy had nine children:

1. Joseph P. Kennedy Jr.
2. John F. Kennedy
3. Rosemary Kennedy
4. Kathleen Kennedy
5. Eunice Kennedy
6. Patricia Kennedy
7. Robert F. Kennedy
8. Jean Kennedy
9. Edward M. Kennedy

The grandchildren of Joseph P. Kennedy include the children of his children. Some of the more well-known grandchildren include Caroline Kennedy (daughter of John F. Kennedy), Maria Shriver (daughter of Eunice Kennedy), and Joseph P. Kennedy II (son of Robert F. Kennedy).


# Few Shot Prompting

In [10]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# Chains

In [11]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | chatModel

res = chain.invoke({"input": "How are you?"})
print(res.content)

¿Cómo estás?


In [12]:
res

AIMessage(content='¿Cómo estás?', response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 52, 'total_tokens': 58, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-715ccd0a-e8e1-4ace-adcb-4a84f5b23724-0', usage_metadata={'input_tokens': 52, 'output_tokens': 6, 'total_tokens': 58})

In [ ]:
# Langchain Expressions Language

# Output Parsers

In [14]:
from langchain_groq import ChatGroq

llmModel = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=groq_api_key,
)

In [13]:
chatModel = llmModel

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an `answer` key that answers the following question: {question}"
)

json_parser = SimpleJsonOutputParser()

json_chain = json_prompt | llmModel | json_parser

In [16]:
json_parser.get_format_instructions()

'Return a JSON object.'

In [17]:
res  = json_chain.invoke({"question": "What is the biggest country?"})

In [18]:
res

{'answer': 'Russia'}

In [22]:
type(res)

dict

#### Optionally, you can use Pydantic to define a custom output format

In [24]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

In [25]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [27]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | chatModel | parser

chain.invoke({"query": "Tell me a joke."})

{'setup': "Why couldn't the bicycle find its way home?",
 'punchline': 'Because it lost its bearings!'}